# Ham ya da Spam?

🎯 Bu görevin amacı, e-postaları **spam (1)** veya **normal e-posta (0)** olarak sınıflandırmaktır.

🧹 İlk olarak, bu metin verilerine **temizleme (cleaning)** teknikleri uygulanacaktır.

👩🏻‍🔬 Ardından, temizlenmiş metinler **sayısal bir gösterime** dönüştürülecektir.

✉️ Son olarak, her bir e-postayı spam mı yoksa normal mi olduğunu sınıflandırmak için  
***Multinomial Naive Bayes*** modeli uygulanacaktır.


## (0) NTLK kütüphanesi (Doğal Dil Araç Seti)

In [1]:
!pip install nltk

In [2]:
import ssl
import certifi

ssl._create_default_https_context = lambda: ssl.create_default_context(cafile=certifi.where())

In [3]:
# nltk'yi ilk kez içe aktarırken, birkaç yerleşik kütüphaneyi de indirmemiz gerekir.

import nltk

nltk.download('stopwords')
nltk.download('punkt')      # nltk<3.9.0 için
nltk.download('punkt_tab')  # nltk>=3.9.0 için
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/buseozgur/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /Users/buseozgur/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/buseozgur/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/buseozgur/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/buseozgur/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [4]:
import pandas as pd

df = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/ham_spam_emails.csv")
df.head()

,text,spam
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1
3,Subject: 4 color printing special request add...,1
4,"Subject: do not have money , get software cds ...",1


## (1) (Metin) veri setinin temizlenmesi

Veri kümesi, ham [0] veya spam [1] olarak sınıflandırılan e-postalardan oluşur. Tahmin modelini eğitmeden önce veri kümesini temizlemeniz gerekir.

### (1.1) Noktalama İşaretlerini Kaldır

❓ Noktalama işaretlerini kaldırmak için bir işlev oluşturun. Bunu `text` sütununa uygulayın ve çıktıyı `clean_text` adlı veri çerçevesinin yeni bir sütununa ekleyin. ❓

In [5]:
# Remove the punctuations
import re

def remove_punctuations(text):
    return re.sub(r'[^\w\s]','',text)

df["clean_text"] = df["text"].apply(remove_punctuations)

df.head(10)

,text,spam,clean_text
0,Subject: naturally irresistible your corporate...,1,Subject naturally irresistible your corporate ...
1,Subject: the stock trading gunslinger fanny i...,1,Subject the stock trading gunslinger fanny is...
2,Subject: unbelievable new homes made easy im ...,1,Subject unbelievable new homes made easy im w...
3,Subject: 4 color printing special request add...,1,Subject 4 color printing special request addi...
4,"Subject: do not have money , get software cds ...",1,Subject do not have money get software cds fr...
5,"Subject: great nnews hello , welcome to medzo...",1,Subject great nnews hello welcome to medzonl...
6,Subject: here ' s a hot play in motion homela...,1,Subject here s a hot play in motion homeland...
7,Subject: save your money buy getting this thin...,1,Subject save your money buy getting this thing...
8,Subject: undeliverable : home based business f...,1,Subject undeliverable home based business for...
9,Subject: save your money buy getting this thin...,1,Subject save your money buy getting this thing...


### (1.2) Küçük Harf

❓ Metni küçük harfe çeviren bir işlev oluşturun. Bunu `clean_text`'e uygulayın ❓

In [6]:
# Lower the letters
def lower_letter(clean_text):
    return clean_text.lower()

df["clean_text"] = df["clean_text"].apply(lower_letter)
df.head(5)

,text,spam,clean_text
0,Subject: naturally irresistible your corporate...,1,subject naturally irresistible your corporate ...
1,Subject: the stock trading gunslinger fanny i...,1,subject the stock trading gunslinger fanny is...
2,Subject: unbelievable new homes made easy im ...,1,subject unbelievable new homes made easy im w...
3,Subject: 4 color printing special request add...,1,subject 4 color printing special request addi...
4,"Subject: do not have money , get software cds ...",1,subject do not have money get software cds fr...


### (1.3) Sayıları Kaldır

❓ Metinden sayıları kaldırmak için bir işlev oluşturun. Bunu `clean_text`'e uygulayın ❓

In [7]:
# Remove the numbers
def remove_numbers(clean_text):
    return re.sub(r'[\d+]','',clean_text)

df['clean_text'] = df['clean_text'].apply(remove_numbers)
df.head(5)

,text,spam,clean_text
0,Subject: naturally irresistible your corporate...,1,subject naturally irresistible your corporate ...
1,Subject: the stock trading gunslinger fanny i...,1,subject the stock trading gunslinger fanny is...
2,Subject: unbelievable new homes made easy im ...,1,subject unbelievable new homes made easy im w...
3,Subject: 4 color printing special request add...,1,subject color printing special request addit...
4,"Subject: do not have money , get software cds ...",1,subject do not have money get software cds fr...


### (1.4) StopWords'ü kaldırın

❓ Metinden durdurma kelimelerini kaldırmak için bir işlev oluşturun. Bunu `clean_text`'e uygulayın. ❓

In [8]:
# Remove stop words
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

stop_words = set(stopwords.words('english')) - {"not", "n't"}

def remove_stopwords(clean_text):
    tokens = word_tokenize(clean_text)
    filtered = [w for w in tokens if w not in stop_words]
    return " ".join(filtered)

df["clean_text"] = df["clean_text"].apply(remove_stopwords)
df.head()

,text,spam,clean_text
0,Subject: naturally irresistible your corporate...,1,subject naturally irresistible corporate ident...
1,Subject: the stock trading gunslinger fanny i...,1,subject stock trading gunslinger fanny merrill...
2,Subject: unbelievable new homes made easy im ...,1,subject unbelievable new homes made easy im wa...
3,Subject: 4 color printing special request add...,1,subject color printing special request additio...
4,"Subject: do not have money , get software cds ...",1,subject not money get software cds software co...


### (1.5) Lemmatize

❓ Metni lemmatize etmek için bir fonksiyon oluşturun. Çıktının bir kelime listesi değil, tek bir dize olduğundan emin olun. Bunu `clean_text`'e uygulayın. ❓

In [ ]:
# Lemmatizze the words
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()
def lemmatize_text(clean_text):
    tokens = word_tokenize(clean_text)
    verbs = [lemmatizer.lemmatize(word, "v") for word in tokens]
    lemmatized = [lemmatizer.lemmatize(word) for word in verbs]
    return " ".join(lemmatized)

df["clean_text"] = df["clean_text"].apply(lemmatize_text)
df.head(5)

,text,spam,clean_text
0,Subject: naturally irresistible your corporate...,1,subject naturally irresistible corporate ident...
1,Subject: the stock trading gunslinger fanny i...,1,subject stock trade gunslinger fanny merrill m...
2,Subject: unbelievable new homes made easy im ...,1,subject unbelievable new home make easy im wan...
3,Subject: 4 color printing special request add...,1,subject color print special request additional...
4,"Subject: do not have money , get software cds ...",1,subject not money get software cd software com...
5,"Subject: great nnews hello , welcome to medzo...",1,subject great nnews hello welcome medzonline s...
6,Subject: here ' s a hot play in motion homela...,1,subject hot play motion homeland security inve...
7,Subject: save your money buy getting this thin...,1,subject save money buy get thing not try ciall...
8,Subject: undeliverable : home based business f...,1,subject undeliverable home base business grown...
9,Subject: save your money buy getting this thin...,1,subject save money buy get thing not try ciall...


## (2) Bag-of-Words Modellemesi

### (2.1) Metin verilerini sayılara dönüştürme

❓ `clean_text`'i varsayılan CountVectorizer ile Bag-of-Words temsiline vektörleştirin. `X_bow` olarak kaydedin. ❓

In [14]:
# Vectorising with BoW
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
X_bow = vectorizer.fit_transform(df["clean_text"])
vectorizer.get_feature_names_out()

array(['aa', 'aaa', 'aaaenerfax', ..., 'zzn', 'zzncacst', 'zzzz'],
      dtype=object)

### (2.2) Çok terimli Naive Bayes Modellemesi

❓ MultinomialNB modelini bag-of-words verileriyle çapraz doğrulayın. Modelin doğruluğunu puanlayın. ❓

In [16]:
# Multinomial Naive Bayes Mdoel
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import cross_val_score

y = df["spam"]
model = MultinomialNB()

scores = cross_val_score(model, X_bow, y, cv=5, scoring= "accuracy")
accuracy = scores.mean()

print("Cross-validated accuracy:", accuracy)

Cross-validated accuracy: 0.9881286723519056


🏁 Tebrikler!

💾 Not defterinizi git add/commit/push yapmayı unutmayın...

🚀 ... ve bir sonraki challenge'a geçin!